Los datos están almacenados en el archivo `/datasets/music_project_en.csv`.



## Etapa 1. Descripción de los datos <a id='data_review'></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/datasets/music_project_en.csv")

Primer vistazo al DataFrame df

In [ ]:
print(df.head(10))

     userID                        Track            artist   genre  \
0  FFB692EC            Kamigata To Boots  The Mass Missile    rock   
1  55204538  Delayed Because of Accident  Andreas Rönnberg    rock   
2    20EC38            Funiculì funiculà       Mario Lanza     pop   
3  A3DD03C9        Dragons in the Sunset        Fire + Ice    folk   
4  E2DC1FAE                  Soul People        Space Echo   dance   
5  842029A1                       Chains          Obladaet  rusrap   
6  4CB90AA5                         True      Roman Messer   dance   
7  F03E1C1F             Feeling This Way   Polina Griffith   dance   
8  8FA1D3BE                     L’estate       Julia Dalia  ruspop   
9  E772D5C0                    Pessimist               NaN   dance   

        City        time        Day  
0  Shelbyville  20:28:33  Wednesday  
1  Springfield  14:07:09     Friday  
2  Shelbyville  20:58:07  Wednesday  
3  Shelbyville  08:37:09     Monday  
4  Springfield  08:34:34     Monday  
5

In [ ]:
# Informacion general
print(df.info)
print()
print(df.isna().sum())

<bound method DataFrame.info of          userID                              Track            artist  \
0      FFB692EC                  Kamigata To Boots  The Mass Missile   
1      55204538        Delayed Because of Accident  Andreas Rönnberg   
2        20EC38                  Funiculì funiculà       Mario Lanza   
3      A3DD03C9              Dragons in the Sunset        Fire + Ice   
4      E2DC1FAE                        Soul People        Space Echo   
...         ...                                ...               ...   
65074  729CBB09                            My Name            McLean   
65075  D08D4A55  Maybe One Day (feat. Black Spade)       Blu & Exile   
65076  C5E3A0D5                          Jalopiina               NaN   
65077  321D0506                      Freight Train     Chas McDevitt   
65078  3A64EF84          Tell Me Sweet Little Lies      Monica Lopez   

            genre       City        time        Day  
0            rock  Shelbyville  20:28:33  Wednesd

Estas son nuestras observaciones sobre la tabla. Contiene siete columnas que almacenan los mismos tipos de datos: `object`.

Según la documentación:
- `' userID'`: identificador del usuario;
- `'Track'`: título de la canción;
- `'artist'`: nombre del artista;
- `'genre'`: género de la canción;
- `'City'`: ciudad del usuario;
- `'time'`: la hora exacta en la que se reprodujo la canción;
- `'Day'`: día de la semana.

Podemos ver dos problemas con el estilo en los encabezados de la tabla:
1. Algunos encabezados están en mayúsculas, otros en minúsculas.
2. Tenemos un error de calidad en los datos, hay muchos valores ausentes o nulos, ademas algunos nombres de las columnas combinan letras mayusculas y minusculas, poseen espacios al inicio, y posiblemente tambien al final.





## Etapa 2. Preprocesamiento de los datos <a id='data_preprocessing'></a>


### Estilo del encabezado <a id='header_style'></a>


In [5]:
# Muestra los nombres de las columnas
print(df.columns)

Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='object')


Vamos cambiar los encabezados de la tabla siguiendo las reglas estilísticas convencionales:
*   Todos los caracteres deben ser minúsculas.
*   Eliminar los espacios.
*   Si el nombre tiene varias palabras, se utiliza snake_case.

In [6]:
# Bucle que itera sobre los encabezados y los pone todos en minúsculas
for col in df.columns:
    col = col.lower()
    print(col)

  userid
track
artist
genre
  city  
time
day


In [7]:
# Bucle que itera sobre los encabezados y elimina los espacios
new_cols = []
for col in df.columns:
    col = col.lower().strip()
    new_cols.append(col)
print(new_cols)
df.columns = new_cols
print(df.columns)
    

['userid', 'track', 'artist', 'genre', 'city', 'time', 'day']
Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


In [ ]:
# Se cambia el nombre de la columna "userid"
print(df.columns)
df = df.rename(columns= {"userid": "user_id"})


Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


In [9]:
# Comprueba el resultado: lista de encabezados
print(df.columns)

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


### Valores ausentes <a id='missing_values'></a>

In [ ]:
# Se calcula el número de valores ausentes
df.isna().sum()

user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64

In [11]:
# Bucle en los encabezados reemplazando los valores ausentes con 'unknown'
columns_to_replace = ['track', 'artist', 'genre']
for column in columns_to_replace:
    df[column].fillna("unknow", inplace=True)

In [ ]:
# Conteo de los valores ausentes
df.isna().sum()

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64

### Duplicados <a id='duplicates'></a>

In [ ]:
# Conteo los duplicados explícitos
df.duplicated().sum()

3826

In [ ]:
# Se eliminan los duplicados explícitos
df = df.drop_duplicates().reset_index(drop=True)

In [15]:
# Comprueba de nuevo si hay duplicados
df.duplicated().sum()

0

Ahora queremos deshacernos de los duplicados implícitos en la columna `genre`. Por ejemplo, el nombre de un género se puede escribir de varias formas. Dichos errores también pueden afectar al resultado.

Etapa 2.11. Primero debemos mostrar una lista de nombres de géneros únicos, por orden alfabético. Para ello:


In [ ]:
# Inspeccion los nombres de géneros únicos
unique =df["genre"].sort_values().unique()
print(unique)


['acid' 'acoustic' 'action' 'adult' 'africa' 'afrikaans' 'alternative'
 'ambient' 'americana' 'animated' 'anime' 'arabesk' 'arabic' 'arena'
 'argentinetango' 'art' 'audiobook' 'avantgarde' 'axé' 'baile' 'balkan'
 'beats' 'bigroom' 'black' 'bluegrass' 'blues' 'bollywood' 'bossa'
 'brazilian' 'breakbeat' 'breaks' 'broadway' 'cantautori' 'cantopop'
 'canzone' 'caribbean' 'caucasian' 'celtic' 'chamber' 'children' 'chill'
 'chinese' 'choral' 'christian' 'christmas' 'classical' 'classicmetal'
 'club' 'colombian' 'comedy' 'conjazz' 'contemporary' 'country' 'cuban'
 'dance' 'dancehall' 'dancepop' 'dark' 'death' 'deep' 'deutschrock'
 'deutschspr' 'dirty' 'disco' 'dnb' 'documentary' 'downbeat' 'downtempo'
 'drum' 'dub' 'dubstep' 'eastern' 'easy' 'electronic' 'electropop' 'emo'
 'entehno' 'epicmetal' 'estrada' 'ethnic' 'eurofolk' 'european'
 'experimental' 'extrememetal' 'fado' 'film' 'fitness' 'flamenco' 'folk'
 'folklore' 'folkmetal' 'folkrock' 'folktronica' 'forró' 'frankreich'
 'französisch' 

Etapa 2.12. Vamos a examinar la lista para identificar **duplicados implícitos** del género `hiphop`, es decir, nombres mal escritos o variantes que hacen referencia al mismo género musical.

Los duplicados que encontrados son:

* `hip`  
* `hop`  
* `hip-hop`  

Para solucionarlo, vamos a crear una función llamada `replace_wrong_values()`.


1. Se define una función llamada `replace_wrong_values()` que recibe los siguientes parámetros:

* `df`: el DataFrame a modificar
* `column`: el nombre de la columna a trabajar
* `wrong_values`: una lista con los valores incorrectos
* `correct_value`: el valor correcto para reemplazar

2. Dentro de la función, se usa un bucle `for` para iterar sobre cada valor incorrecto y aplicar `.replace()`.


In [17]:
# Función para reemplazar los duplicados implícitos
def replace_wrong_values(df, column, wrong_values, correct_value):
    for value in wrong_values:
        df[column].replace(value, correct_value, inplace=True)

In [ ]:
# Se eliminan los duplicados implícitos
replace_wrong_values(df, "genre", ['hip', 'hop', 'hip-hop'], 'hiphop')

In [ ]:
# Se validan de nuevo los duplicados implícitos
df["genre"].sort_values().unique()

array(['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans',
       'alternative', 'ambient', 'americana', 'animated', 'anime',
       'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook',
       'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom',
       'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian',
       'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop',
       'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber',
       'children', 'chill', 'chinese', 'choral', 'christian', 'christmas',
       'classical', 'classicmetal', 'club', 'colombian', 'comedy',
       'conjazz', 'contemporary', 'country', 'cuban', 'dance',
       'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock',
       'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat',
       'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy',
       'electronic', 'electropop', 'emo', 'entehno', 'epicmetal',
       'estrada', 'ethnic', 'eurofo

## Etapa 3. Análisis

### Comparar el comportamiento de los usuarios en las dos ciudades <a id='activity'></a>

Queremos analizar si hay diferencias en la cantidad de canciones reproducidas en Springfield y Shelbyville. Para ello, usaremos los datos de dos días de la semana: lunes y viernes.

Compararemos cuántas canciones se escucharon en cada ciudad durante esos días para identificar posibles patrones de comportamiento.

Se siguen estos tres pasos para organizar el análisis:

- Dividir: agrupar los datos por ciudad.

- Aplicar: contar cuántas canciones se reproducen en cada grupo.

- Combinar: se presentan los resultados de forma que se puedan comparar fácilmente ambas ciudades.


In [20]:
# Cuenta las canciones reproducidas en cada ciudad
track_df =df.groupby(by="city")["track"].count()
print(track_df)

city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64




¿Qué diferencias se  encontraron entre Springfield y Shelbyville? ¿A qué podrían deberse?

Hay una diferencia significativa de canciones reproducidas entre las dos ciudades, siendo Springfield la que mayor reproducciones tiene, mas del doble que Shelbyville. Esto podria deberse a una mayor actividad en la ciudad de Springfield, o una poblacion mayor en compracion a Shelyville.


In [ ]:
# Se calculan las canciones reproducidas en cada uno de los dos días
days = ["Monday", "Friday"]
filtered_df = df.query("day in @days")
track_df = filtered_df.groupby(by="day")["track"].count()
print(track_df)
print()

#friday_df = df.query("day in@days[1]")
#track_friday_df  = friday_df.groupby(by="day")["track"].count()
#print(track_friday_df)
#print()

#monday_df = df.query("day in@days[0]")
#track_monday_df  = monday_df.groupby(by="day")["track"].count()
#print(track_monday_df)


day
Friday    21840
Monday    21354
Name: track, dtype: int64

day
Friday    21840
Name: track, dtype: int64

day
Monday    21354
Name: track, dtype: int64




¿Hubo un día con más actividad? ¿Cambia algo si analizas cada ciudad por separado?



La diferencia entre las canciones reproducidas se reduce significativamente, los resultados no son tan diferentes, aunque el dia que mas reproducciones hubo fue el viernes
Y analizar los datos por separado no represneto ninguna diferencia.

Etapa 3.5

Ahora vamos a combinar dos criterios: día y ciudad.

Se crea una función llamada `number_tracks()` que reciba dos parámetros:

* `day`: un día de la semana (por ejemplo, `'Monday'`)
* `city`: el nombre de una ciudad (por ejemplo, `'Springfield'`)

Dentro de la función:

1. Filtra el DataFrame por el día.
2. Luego, filtra por la ciudad.
3. Cuenta cuántas veces aparece `'user_id'` en ese filtro.
4. Devuelve ese número como resultado.


In [ ]:
# Se declara la función number_tracks() con dos parámetros: day= y city=.
def number_tracks(day, city):
    day_df = df[df["day"] == day]
    city_df = day_df[day_df["city"]==city]
    times_user_id = city_df["user_id"].count()
    return times_user_id
    # Almacena las filas del DataFrame donde el valor en la columna 'day' es igual al parámetro day=

    # Filtra las filas donde el valor en la columna 'city' es igual al parámetro city=

    # Extrae la columna 'user_id' de la tabla filtrada y aplica el método count()

    # Devuelve el número de valores de la columna 'user_id'

Etapa 3.6. Llama a `number_tracks()` cuatro veces: una por ciudad en cada uno de los dos días.

In [23]:
# El número de canciones reproducidas en Springfield el lunes
number_tracks("Monday", "Springfield")

15740

In [24]:
# El número de canciones reproducidas en Shelbyville el lunes
number_tracks("Monday", "Shelbyville")

5614

In [25]:
# El número de canciones reproducidas en Springfield el viernes
number_tracks("Friday", "Springfield")

15945

In [26]:
# El número de canciones reproducidas en Shelbyville el viernes
number_tracks("Friday", "Shelbyville")

5895

# Conclusiones <a id='end'></a>

## Escribe tus conclusiones finales sobre el análisis

Pregunta central:

> *¿Los datos muestran que el comportamiento de los usuarios —en cuanto a la música que escuchan— varía según la ciudad y el día de la semana?*




Uno de los principales patrones que se encontrar es que la actividad en Springfield es mucho mayor a la de Shelbyville, casi del 300% mayor tanto en la actividad total como en la actividad por dia para el lunes y viernes.

El DataFrame presentaba algunos problemas con la calidad de los datos, tuvimos problemas de consistencia con los nombres de las columnas, pues estos combinaban mayusculas y minusculas y ademas tenian espacios antes o al final del nombre de la columna. Para solucionar esto tuvimos que cambiar los nombres de las columnas a letras minusculas, borrar los espacios al inicio o final del string y separar "userid" a "user_id".

También se encontraron problemas de completitud como valores ausentes en el DataFrame, a los cuales se les tuvo que asignar el valor "unknow" en las columnas 'track', 'artist' y 'genre'. 
Se identificaron problemas de presicion con algunos valores repetidos que tuvieron que eliminarse con el método .drop_duplicates(). 
Y otro de los problemas fue el de duplicados implicitos dentro de la columna "genre", los cuales fueron tratados para obtener solamente el genero hiphop.

Hacer todo lo anterior nos ayudó a obtener resultados mas acertados para el analisis, ademas de otorgarnos muchisima mas facilidad para poder manipular los datos, filtrar de manera rapida y conscisa y poder aplicar métodos y funciones a los datos.